In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from transformer_lens import HookedTransformer
import torch

import json
import time
import os

from src.utils import tuple_str_to_tuple
from src.neuron_texts import get_neuron_max_acts

In [3]:
# Load jsons from ../experiment_data/1_next_token_neurons
# filename = "2023-10-11_19-25-40_gpt2-xl"
# filename = "2023-10-11_19-07-38_gpt2-small"
# filename = "2023-10-11_19-19-59_gpt2-medium"
# filename = "2023-10-13_17-59-51_gpt2-large"

filename = "2026-06-15_19-06-52_gpt2-small_random"
is_random = "-random" # otherwise ""
#filename = "2026-05-09_16-12-03_gpt2-medium"

# filename = "2024-02-13_04-33-18_pythia-1.4b"
# filename = "2024-02-15_07-29-20_pythia-410m"

train = False

with open(f'../experiment_data/1_next_token_neurons/{filename}.json') as f:
    neurons_data = json.load(f)

In [4]:
# Parameters
parameters = neurons_data['parameters']
model_name = parameters['model_name']
neurons_list = [tuple_str_to_tuple(x) for x in neurons_data['neurons'].keys()]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
model = HookedTransformer.from_pretrained(
    model_name,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    # refactor_factored_attn_matrices=True,
    device=device,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [6]:
import pickle
dataset_text_test_path = "../experiment_data/text_dict_test.pkl"
dataset_text_train_path = "../experiment_data/text_dict_train.pkl"
if train:
    with open(dataset_text_train_path, 'rb') as f:
        dataset = pickle.load(f)
    dataset_text_list = [x['text'] for x in dataset]
    print("Train loaded!")

else:
    with open(dataset_text_test_path, 'rb') as f:
        dataset = pickle.load(f)

    dataset_text_list = [x['text'] for x in dataset]

    # removing samples that were also in the training dataset
    dataset_text_list.pop(9272) 
    dataset_text_list.pop(7138)
    print("Test loaded!")

Test loaded!


In [7]:
#!export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [8]:
with torch.no_grad():
    neuron_max_acts = get_neuron_max_acts(
        model=model,
        dataset_text_list=dataset_text_list,
        neurons_list=neurons_list,
        batch_size=1,
        device=device,
    )


100%|██████████| 9998/9998 [09:43<00:00, 17.13it/s]


In [9]:
output = {
    'parameters': parameters,
    'neuron_max_acts': {str(key): value for key, value in neuron_max_acts.items()},
    'prior_filename': filename
}

# Save json to ../experiment_data/2_max_activating_texts
timestamp = time.strftime("%Y-%m-%d_%H-%M-%S", time.localtime(int(time.time())))
train_str = "_train" if train else "_test"
new_filename = f"{timestamp}_{model_name}{is_random}{train_str}.json"

folder_path = "../experiment_data/2_max_activating_texts"
filepath = os.path.join(folder_path, new_filename)
if os.path.isfile(filepath):
    raise Exception("File already exists!")
os.makedirs(folder_path, exist_ok=True)

with open(filepath, 'w') as f:
    json.dump(output, f)

In [10]:
print(len(neuron_max_acts))

60
